In [6]:
import requests
import pandas as pd
import json

In [7]:
# PepsiCo CIK number on SEC EDGAR
CIK = "0000077476"

# Fetch all company facts from SEC EDGAR
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json"
headers = {"User-Agent": "Fatiha Okesola fatihaayink@gmail.com"}

response = requests.get(url, headers=headers)
data = response.json()

print("Status:", response.status_code)
print("Company:", data["entityName"])

Status: 200
Company: PepsiCo, Inc.


In [8]:
# Navigate to the US-GAAP facts
gaap = data["facts"]["us-gaap"]

# Print available keys to see what financial line items are available
print(f"Total financial line items available: {len(gaap)}")
print("\nSample line items:")
for key in list(gaap.keys())[:20]:
    print(f"  {key}")

Total financial line items available: 612

Sample line items:
  AccountsNotesAndLoansReceivableNetCurrent
  AccountsPayableAndAccruedLiabilitiesCurrent
  AccountsPayableCurrent
  AccountsReceivableGrossCurrent
  AccruedEmployeeBenefitsCurrent
  AccruedIncomeTaxesCurrent
  AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment
  AccumulatedOtherComprehensiveIncomeLossAvailableForSaleSecuritiesAdjustmentNetOfTax
  AccumulatedOtherComprehensiveIncomeLossCumulativeChangesInNetGainLossFromCashFlowHedgesEffectNetOfTax
  AccumulatedOtherComprehensiveIncomeLossDefinedBenefitPensionAndOtherPostretirementPlansNetOfTax
  AccumulatedOtherComprehensiveIncomeLossForeignCurrencyTranslationAdjustmentNetOfTax
  AccumulatedOtherComprehensiveIncomeLossNetOfTax
  AdditionalPaidInCapitalCommonStock
  AdjustmentsToAdditionalPaidInCapitalTaxEffectFromShareBasedCompensation
  AdvertisingExpense
  AllocatedShareBasedCompensationExpense
  AllowanceForDoubtfulAccountsReceivableCurrent
  Amortiz

In [9]:
# Search for the specific tags we need
search_terms = ['revenue', 'netincome', 'grossprofit', 'assets', 'equity', 'costof']

print("Searching for relevant tags:\n")
for term in search_terms:
    matches = [key for key in gaap.keys() if term.lower() in key.lower()]
    print(f"--- '{term}' ---")
    for m in matches[:5]:
        print(f"  {m}")
    print()

Searching for relevant tags:

--- 'revenue' ---
  BusinessAcquisitionProFormaRevenue
  BusinessAcquisitionsProFormaRevenue
  Revenues
  SalesRevenueNet

--- 'netincome' ---
  BusinessAcquisitionProFormaNetIncomeLoss
  BusinessAcquisitionsProFormaNetIncomeLoss
  NetIncomeLoss
  NetIncomeLossAttributableToNoncontrollingInterest
  NetIncomeLossAvailableToCommonStockholdersBasic

--- 'grossprofit' ---
  GrossProfit

--- 'assets' ---
  AmortizationOfIntangibleAssets
  Assets
  AssetsCurrent
  AssetsOfDisposalGroupIncludingDiscontinuedOperation
  BusinessAcquisitionPurchasePriceAllocationAmortizableIntangibleAssets

--- 'equity' ---
  AvailableForSaleEquitySecuritiesAccumulatedGrossUnrealizedGainBeforeTax
  AvailableForSaleEquitySecuritiesGrossUnrealizedGain
  BusinessAcquisitionCostOfAcquiredEntityEquityInterestsIssuedAndIssuable
  BusinessCombinationStepAcquisitionEquityInterestInAcquireeRemeasurementGainOrLoss
  DebtAndEquitySecuritiesRealizedGainLoss

--- 'costof' ---
  BusinessAcquisiti

In [10]:
# Search specifically for stockholders equity
equity_matches = [key for key in gaap.keys() 
                  if 'stockholder' in key.lower() or 
                  'shareholdersEquity' in key.lower() or
                  'StockholdersEquity' in key]

print("Equity tags found:")
for m in equity_matches:
    print(f"  {m}")

Equity tags found:
  LiabilitiesAndStockholdersEquity
  NetIncomeLossAvailableToCommonStockholdersBasic
  NetIncomeLossAvailableToCommonStockholdersDiluted
  StockholdersEquity
  StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest


In [11]:
def extract_annual(tag, label):
    """
    This function pulls annual figures for one financial line item.
    
    tag: the XBRL tag name (e.g. 'Revenues')
    label: a readable name we give it (e.g. 'Net Revenue')
    
    How it works:
    - gaap[tag] gets the data for that specific tag
    - ['units'] gets the unit of measurement (USD, shares, etc.)
    - ['USD'] gets the dollar-denominated values
    - We then filter to keep only annual figures (form 10-K)
      and only the years we want (2019-2025)
    """
    try:
        # Get all reported values for this tag in USD
        entries = gaap[tag]['units']['USD']
        
        # Keep only annual 10-K filings, not quarterly
        # 'form' tells us if it's a 10-K or 10-Q
        annual = [e for e in entries if e['form'] == '10-K']
        
        # Keep only fiscal years 2019 through 2025
        # 'fy' is the fiscal year the number belongs to
        filtered = [e for e in annual if 2019 <= e['fy'] <= 2025]
        
        # Remove duplicates — sometimes the same year appears
        # multiple times due to restatements or amendments
        # We keep the most recently filed version using 'filed' date
        seen = {}
        for e in filtered:
            yr = e['fy']
            if yr not in seen or e['filed'] > seen[yr]['filed']:
                seen[yr] = e
        
        # Build a clean dictionary of year: value
        result = {yr: entry['val'] for yr, entry in seen.items()}
        return result
    
    except KeyError:
        # If the tag doesn't exist, return empty
        print(f"Tag not found: {tag}")
        return {}

# Test it on revenue first
revenue = extract_annual('Revenues', 'Net Revenue')
print("Net Revenue by year:")
for year, val in sorted(revenue.items()):
    print(f"  {year}: ${val/1e9:.2f}B")

Net Revenue by year:
  2019: $63.52B
  2020: $64.66B
  2021: $67.16B
  2022: $70.37B
  2023: $79.47B
  2024: $86.39B
  2025: $91.47B


In [12]:
# Let's check the raw entries to see exact dates
# This helps us verify fiscal year labelling is correct
entries = gaap['Revenues']['units']['USD']
annual = [e for e in entries if e['form'] == '10-K']
recent = [e for e in annual if e['fy'] >= 2019]

print("Raw annual revenue entries:")
for e in sorted(recent, key=lambda x: x['fy']):
    print(f"  FY:{e['fy']} | Period end:{e['end']} | Filed:{e['filed']} | Value:${e['val']/1e9:.2f}B")

Raw annual revenue entries:
  FY:2019 | Period end:2017-12-30 | Filed:2020-02-13 | Value:$63.52B
  FY:2019 | Period end:2018-03-24 | Filed:2020-02-13 | Value:$12.56B
  FY:2019 | Period end:2018-06-16 | Filed:2020-02-13 | Value:$16.09B
  FY:2019 | Period end:2018-09-08 | Filed:2020-02-13 | Value:$16.48B
  FY:2019 | Period end:2018-12-29 | Filed:2020-02-13 | Value:$64.66B
  FY:2019 | Period end:2018-12-29 | Filed:2020-02-13 | Value:$19.52B
  FY:2019 | Period end:2019-03-23 | Filed:2020-02-13 | Value:$12.88B
  FY:2019 | Period end:2019-06-15 | Filed:2020-02-13 | Value:$16.45B
  FY:2019 | Period end:2019-09-07 | Filed:2020-02-13 | Value:$17.19B
  FY:2019 | Period end:2019-12-28 | Filed:2020-02-13 | Value:$67.16B
  FY:2019 | Period end:2019-12-28 | Filed:2020-02-13 | Value:$20.64B
  FY:2020 | Period end:2018-12-29 | Filed:2021-02-11 | Value:$64.66B
  FY:2020 | Period end:2019-12-28 | Filed:2021-02-11 | Value:$67.16B
  FY:2020 | Period end:2020-12-26 | Filed:2021-02-11 | Value:$70.37B
  FY:2

## Note on EDGAR Fiscal Year Labels

The initial extraction used EDGAR's `fy` field to filter by fiscal year. 
Inspecting the raw entries revealed that each 10-K filing includes three 
years of comparative data, all tagged with the same `fy` value — causing 
duplicates and misaligned years. 

The corrected approach filters by `period end date` instead, which 
accurately identifies which fiscal year each value belongs to. This 
matches PepsiCo's reported figures in the 10-K filings.

In [13]:
def extract_by_period(tag):
    """
    Extract annual figures by period end date rather than fiscal year label.
    This avoids the duplication issue where each 10-K reports 3 years of data.
    
    We filter to keep only:
    - Annual 10-K filings (not quarterly)
    - Period end dates falling in our analysis window (2019-2025)
    - One entry per period end date — the most recently filed version
    """
    try:
        entries = gaap[tag]['units']['USD']
        
        # Keep only 10-K annual filings
        annual = [e for e in entries if e['form'] == '10-K']
        
        # Filter by period end date year, not fy label
        # Period end dates like 2019-12-28 belong to fiscal year 2019
        filtered = [e for e in annual 
                   if 2019 <= int(e['end'][:4]) <= 2025]
        
        # Keep only full year entries — period must end in Nov or Dec
        # This removes quarterly sub-periods that slip through
        filtered = [e for e in filtered 
                   if int(e['end'][5:7]) >= 11]
        
        # Remove duplicates — keep most recently filed per period end
        seen = {}
        for e in filtered:
            end = e['end']
            if end not in seen or e['filed'] > seen[end]['filed']:
                seen[end] = e
        
        # Map period end year to value
        result = {int(end[:4]): entry['val'] 
                 for end, entry in seen.items()}
        return result
    
    except KeyError:
        print(f"Tag not found: {tag}")
        return {}

# Test with revenue
revenue = extract_by_period('Revenues')
print("Net Revenue by fiscal year:")
for year, val in sorted(revenue.items()):
    print(f"  {year}: ${val/1e9:.2f}B")

Net Revenue by fiscal year:
  2019: $67.16B
  2020: $70.37B
  2021: $79.47B
  2022: $86.39B
  2023: $91.47B
  2024: $91.85B
  2025: $93.92B


In [14]:
# Define all the tags we want to extract
# Each tuple is (XBRL tag, readable label)
tags = [
    ('Revenues', 'Net Revenue'),
    ('CostOfGoodsAndServicesSold', 'Cost of Sales'),
    ('GrossProfit', 'Gross Profit'),
    ('NetIncomeLoss', 'Net Income'),
    ('Assets', 'Total Assets'),
    ('StockholdersEquity', 'Shareholders Equity'),
]

# Extract all items and store in a dictionary
# Key is the readable label, value is a dict of {year: amount}
pepsico_financials = {}
for tag, label in tags:
    pepsico_financials[label] = extract_by_period(tag)

# Convert to a pandas dataframe
# Rows = financial line items, Columns = fiscal years
pepsico_financials_df = pd.DataFrame(pepsico_financials).T

# Sort columns by year (left to right: 2019 to 2025)
pepsico_financials_df = pepsico_financials_df[sorted(pepsico_financials_df.columns)]

# Divide by 1 billion for readability
pepsico_financials_billions = (pepsico_financials_df / 1e9).round(2)

print("PepsiCo Financial Summary ($ billions):")
print(pepsico_financials_billions.to_string())

PepsiCo Financial Summary ($ billions):
                      2019   2020   2021   2022    2023   2024    2025
Net Revenue          67.16  70.37  79.47  86.39   91.47  91.85   93.92
Cost of Sales        30.13  31.80  37.08  40.58   41.88  41.74   43.07
Gross Profit         37.03  38.58  42.40  45.82   49.59  50.11   50.86
Net Income            7.31   7.12   7.62   8.91    9.07   9.58    8.24
Total Assets         78.55  92.92  92.38  92.19  100.50  99.47  107.40
Shareholders Equity  14.79  13.45  16.04  17.15   18.50  18.04   20.41


In [15]:
# Save the extracted data to Excel
# We write to a new sheet called 'Raw Data' in your existing workbook
# if_sheet_exists='replace' means if the sheet already exists, overwrite it

output_path = r'C:\Users\fatih\OneDrive\Documents\My Portfolio Projects\Pepsico Analysis Series\PepsiCo_FSA.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    pepsico_financials_df.to_excel(writer, sheet_name='Raw Data')

print("Successfully saved to PepsiCo_FSA.xlsx — Raw Data sheet")

Successfully saved to PepsiCo_FSA.xlsx — Raw Data sheet


In [16]:
# Save in millions format — standard for financial statements
# Divide by 1,000,000 and round to 1 decimal place
pepsico_financials_millions = (pepsico_financials_df / 1e6).round(1)

with pd.ExcelWriter('PepsiCo_FSA.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    pepsico_financials_millions.to_excel(writer, sheet_name='Raw Data')

print("Successfully saved Raw Data in millions format")

Successfully saved Raw Data in millions format
